<a href="https://colab.research.google.com/github/dhruv-verma-sys/geostorm/blob/main/ML_exp_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install dash plotly pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 26.7 MB/s eta 0:00:00


In [2]:
%%writefile app.py

import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

# Load the data
data = pd.read_csv(
    'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/historical_automobile_sales.csv'
)

# Initialize Dash application
app = dash.Dash(__name__)

# Application title
app.title = "Automobile Sales Statistics Dashboard"

# Dropdown options
dropdown_options = [
    {
        'label': 'Yearly Statistics',
        'value': 'Yearly Statistics'
    },
    {
        'label': 'Recession Period Statistics',
        'value': 'Recession Period Statistics'
    }
]

# List of years
year_list = [i for i in range(1980, 2024)]


# Layout
app.layout = html.Div([

    html.H1(
        "Automobile Sales Statistics Dashboard",
        style={
            'textAlign': 'center',
            'color': '#503D36',
            'font-size': 24
        }
    ),

    html.Div([
        html.Label("Select Statistics:"),

        dcc.Dropdown(
            id='dropdown-statistics',
            options=dropdown_options,
            value='Yearly Statistics',
            placeholder='Select a report type'
        )
    ]),

    html.Div([
        html.Label("Select Year:"),

        dcc.Dropdown(
            id='select-year',
            options=[
                {'label': i, 'value': i}
                for i in year_list
            ],
            value=1980,
            placeholder='Select a year'
        )
    ]),

    html.Div(
        id='output-container',
        className='output-container'
    )
])


# Callback for year dropdown
@app.callback(
    Output(
        component_id='select-year',
        component_property='disabled'
    ),
    Input(
        component_id='dropdown-statistics',
        component_property='value'
    )
)

def update_input_container(selected_statistics):

    if selected_statistics == 'Recession Period Statistics':
        return True
    else:
        return False


# Callback for graphs
@app.callback(
    Output(
        component_id='output-container',
        component_property='children'
    ),
    [
        Input(
            component_id='select-year',
            component_property='value'
        ),
        Input(
            component_id='dropdown-statistics',
            component_property='value'
        )
    ]
)

def update_output_container(input_year, selected_statistics):

    # -----------------------------------------
    # RECESSION REPORT
    # -----------------------------------------

    if selected_statistics == 'Recession Period Statistics':

        recession_data = data[data['Recession'] == 1]

        # Plot 1
        yearly_rec = recession_data.groupby(
            'Year'
        )['Automobile_Sales'].mean().reset_index()

        R_chart1 = dcc.Graph(
            figure=px.line(
                yearly_rec,
                x='Year',
                y='Automobile_Sales',
                title='Average Automobile Sales Fluctuation Over Recession Period'
            )
        )

        # Plot 2
        average_sales = recession_data.groupby(
            'Vehicle_Type'
        )['Automobile_Sales'].mean().reset_index()

        R_chart2 = dcc.Graph(
            figure=px.bar(
                average_sales,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title='Average Automobile Sales by Vehicle Type During Recession'
            )
        )

        # Plot 3
        exp_data = recession_data.groupby(
            'Vehicle_Type'
        )['Advertising_Expenditure'].sum().reset_index()

        R_chart3 = dcc.Graph(
            figure=px.pie(
                exp_data,
                values='Advertising_Expenditure',
                names='Vehicle_Type',
                title='Advertisement Expenditure by Vehicle Type During Recession'
            )
        )

        # Plot 4
        unemp_data = recession_data.groupby(
            ['Vehicle_Type', 'unemployment_rate']
        )['Automobile_Sales'].mean().reset_index()

        R_chart4 = dcc.Graph(
            figure=px.bar(
                unemp_data,
                x='unemployment_rate',
                y='Automobile_Sales',
                color='Vehicle_Type',
                title='Effect of Unemployment Rate on Vehicle Type and Sales'
            )
        )

        return [

            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=R_chart1),
                    html.Div(children=R_chart2)
                ]
            ),

            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=R_chart3),
                    html.Div(children=R_chart4)
                ]
            )
        ]


    # -----------------------------------------
    # YEARLY REPORT
    # -----------------------------------------

    elif (
        input_year
        and selected_statistics == 'Yearly Statistics'
    ):

        yearly_data = data[data['Year'] == input_year]

        # Plot 1
        yas = data.groupby(
            'Year'
        )['Automobile_Sales'].mean().reset_index()

        Y_chart1 = dcc.Graph(
            figure=px.line(
                yas,
                x='Year',
                y='Automobile_Sales',
                title='Yearly Automobile Sales'
            )
        )

        # Plot 2
        monthly_sales = yearly_data.groupby(
            'Month'
        )['Automobile_Sales'].sum().reset_index()

        Y_chart2 = dcc.Graph(
            figure=px.line(
                monthly_sales,
                x='Month',
                y='Automobile_Sales',
                title='Monthly Automobile Sales in {}'.format(input_year)
            )
        )

        # Plot 3
        average_vehicle_sales = yearly_data.groupby(
            'Vehicle_Type'
        )['Automobile_Sales'].mean().reset_index()

        Y_chart3 = dcc.Graph(
            figure=px.bar(
                average_vehicle_sales,
                x='Vehicle_Type',
                y='Automobile_Sales',
                title='Average Vehicles Sold by Vehicle Type in {}'.format(input_year)
            )
        )

        # Plot 4
        exp_data = yearly_data.groupby(
            'Vehicle_Type'
        )['Advertising_Expenditure'].sum().reset_index()

        Y_chart4 = dcc.Graph(
            figure=px.pie(
                exp_data,
                values='Advertising_Expenditure',
                names='Vehicle_Type',
                title='Total Advertisement Expenditure by Vehicle Type in {}'.format(input_year)
            )
        )

        return [

            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=Y_chart1),
                    html.Div(children=Y_chart2)
                ]
            ),

            html.Div(
                className='chart-item',
                children=[
                    html.Div(children=Y_chart3),
                    html.Div(children=Y_chart4)
                ]
            )
        ]

    else:
        return None


# Run application
if __name__ == '__main__':
    app.run(debug=True)

Writing app.py


In [3]:
import os

print(os.path.exists("app.py"))

True


In [4]:
!python -m py_compile app.py

In [5]:
!python app.py

Dash is running on http://127.0.0.1:8050/

 * Serving Flask app 'app'
 * Debug mode: on


In [ ]:
app = dash.Dash(__name__)

app.title = "Automobile Sales Statistics Dashboard"

In [ ]:
html.H1(
    "Automobile Sales Statistics Dashboard"
)

In [ ]:
dcc.Dropdown(
    id='dropdown-statistics',
    options=dropdown_options,
    value='Yearly Statistics'
)

In [ ]:
dcc.Dropdown(
    id='select-year',
    options=[
        {'label': i, 'value': i}
        for i in year_list
    ]
)

In [ ]:
html.Div(
    id='output-container',
    className='output-container'
)